# SOC Agent — Standard of Care Analysis

In [ ]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from agent_prompt_functions import get_soc_agent_prompt
from data_utils import filterSOCLOT, formatSOCLotDF

print(f'CT_ROOT : {CT_ROOT}')

In [ ]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

In [ ]:
DATA_FILE = CT_ROOT / 'data' / 'rwd_lot.xlsx'

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
print(f'Sheets found: {list(sheets.keys())}')

# Adjust sheet name to match your workbook
soc_raw = sheets['soc']

filtered_rows, topN_per_line = filterSOCLOT(soc_raw, pct=0.03, N=5)
soc_df = formatSOCLotDF(topN_per_line)

print(f'Rows after filterSOCLOT (top-N): {len(topN_per_line)}')
print(soc_df[:500])

In [ ]:
# Paste I/E criteria here
ie_criteria = ""


In [ ]:
soc_prompt = get_soc_agent_prompt(ie_criteria, soc_df)
print(f'Prompt length: {len(soc_prompt)} chars')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API
client = anthropic.Anthropic()

response = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=4096,
    messages=[{'role': 'user', 'content': soc_prompt}],
)

display(Markdown(response.content[0].text))